## Implementazione di un Bot Discord per la Classificazione CIFAR-10
In questa sezione integriamo il modello addestrato con un Bot Discord. 
Il bot riceverà un'immagine, la processerà e restituirà la classe predetta.

In [1]:
# Installazione delle librerie necessarie
#!pip install discord.py tensorflow pillow nest-asyncio

import nest_asyncio
import io
import numpy as np
import tensorflow as tf
from PIL import Image
import discord
from discord.ext import commands
import os
from dotenv import load_dotenv 

# Patch necessaria per far girare il loop di Discord dentro Jupyter
nest_asyncio.apply()  #Ho incluso nest_asyncio perché i file Jupyter hanno spesso conflitti con i loop di Discord.

In [2]:
import os
import discord
from discord.ext import commands
from dotenv import load_dotenv 

load_dotenv('TOKEN')
TOKEN = os.getenv('TOKEN')

##### 1. Caricamento del Modello e Definizione Classi
Carichiamo il file `.h5` generato durante la fase di training e definiamo le 10 etichette del dataset CIFAR-10.

In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

MODEL_PATH = r'C:\Users\Mary_Rosy\Desktop\gitdesktop\Progetto_Finale_Gruppo3\RESULT\cifar10_improved_model.keras'


try:
    # Aggiungiamo compile=False per ignorare l'ottimizzatore PyTorch incompatibile
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print("✅ Modello caricato correttamente (modalità inferenza)!")
    
    # Visualizziamo la struttura per essere sicuri che sia integro
    model.summary()
    
except Exception as e:
    print(f"❌ Errore nel caricamento: {e}")

class_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 
               'cane', 'rana', 'cavallo', 'nave', 'camion']

✅ Modello caricato correttamente (modalità inferenza)!


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,29

 Total params: 404,778 (1.54 MB)

 Trainable params: 403,882 (1.54 MB)

 Non-trainable params: 896 (3.50 KB)

# Preprocessing

## 2. Funzione di Pre-processing
Le immagini inviate su Discord possono avere qualsiasi dimensione. La funzione `prepare_image` si occupa di:
1. Convertire l'immagine in RGB.
2. Ridimensionarla a 32x32 pixel.
3. Normalizzare i valori dei pixel tra 0 e 1.

In [4]:
def prepare_image(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((32, 32), Image.Resampling.LANCZOS)
    # Normalizzazione [0, 1] coerente con la cella 5 di CNN_base
    img_array = np.array(img).astype('float32') / 255.0
    return np.expand_dims(img_array, axis=0)

# Configurazione e Comandi Bot

### 3. Configurazione del Bot Discord
Definiamo il prefisso del comando (`!`) e la logica per gestire l'allegato. 
Il bot scaricherà l'immagine, userà il modello per la predizione e risponderà all'utente.

In [5]:
# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)

In [6]:
class_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 'cane', 'rana', 'cavallo', 'nave', 'camion']

In [ ]:
@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Allega una foto!")
        return

    attachment = ctx.message.attachments[0]
    
    # 1. Messaggio iniziale (Stato t=0)
    loading_msg = await ctx.send("⌛ Inizializzazione scansione...")
    
    try:
        # 2. Simulazione progresso visivo (Fase di pre-processing)
        # Questo mima la latenza di caricamento del segnale
        await loading_msg.edit(content="📡 `███░░░░░░░` Acquisizione segnale...")
        
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        # 3. Simulazione progresso (Fase di inferenza)
        await loading_msg.edit(content="🧠 `███████░░░` Analisi neurale in corso...")
        
        # Esecuzione effettiva del modello
        preds = model(processed_img, training=False)
        probs = preds[0].numpy()
        index = np.argmax(probs)
        confidenza = probs[index] * 100
        
        # 4. Completamento scansione
        await loading_msg.edit(content="✅ `██████████` Scansione completata!")
        
        # Creazione dell'Embed finale
        embed = discord.Embed(title="🔍 Analisi CIFAR-10", color=0x3498db)
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Identificato", value=f"✨ **{class_names[index].upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza:.2f}%", inline=True)
        
        embed.set_footer(text=f"AI Model: image recognizer bot | Crediti: MaGMI")

        # Rimuoviamo il messaggio di caricamento e inviamo il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore durante la scansione: {e}")

### Gestione Errori

In [8]:
# 1. Configurazione Stato Personalizzato
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !info"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!info")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot online come: {bot.user}')

# 2. Gestione Errori Globale
@bot.event
async def on_command_error(ctx, error):
    if isinstance(error, commands.CommandNotFound):
        await ctx.send("❓ Comando non riconosciuto. Scrivi `!info` per vedere cosa posso fare.")
    elif isinstance(error, commands.MissingRequiredArgument):
        await ctx.send("⚠️ Mancano degli argomenti nel comando.")
    else:
        print(f"Errore non gestito: {error}")

### Comando Informativo
Aggiungiamo un comando per spiegare agli utenti quali sono le 10 categorie che il modello può riconoscere (dataset CIFAR-10) e come utilizzare il bot.

In [9]:
@bot.command()
async def info(ctx):
    # Creiamo un messaggio formattato in modo elegante (Embed)
    descrizione = (
        "Ciao! Sono il Bot del progetto di Classificazione Immagini.\n"
        "Utilizzo una rete neurale (CNN) addestrata sul dataset **CIFAR-10**.\n\n"
        "**Cosa posso fare?**\n"
        "Se mi invii una foto e scrivi `!classifica`, proverò a capire cosa rappresenta.\n\n"
        "**Le mie 10 categorie sono:**\n"
        f"✈️ {class_names[0]}, 🚗 {class_names[1]}, 🐦 {class_names[2]}, 🐱 {class_names[3]}, 🦌 {class_names[4]},\n"
        f"🐶 {class_names[5]}, 🐸 {class_names[6]}, 🐴 {class_names[7]}, 🚢 {class_names[8]}, 🚛 {class_names[9]}.\n\n"
        "**Istruzioni:**\n"
        "Carica una foto e scrivi `!classifica` nel campo del commento!"
    )
    
    await ctx.send(descrizione)

# Esecuzione
Inserisci il tuo Token e avvia la cella. Il bot rimarrà attivo finché la cella è in esecuzione.

In [10]:
bot.run(TOKEN)

[2026-03-12 10:33:38] [INFO    ] discord.client: logging in using static token
[2026-03-12 10:33:40] [INFO    ] discord.gateway: Shard ID None has connected to Gateway (Session ID: 83da4627aecf5665ab7479e3a078cd8d).


✅ Bot online come: image recognizer bot#6774


[2026-03-12 11:12:09] [INFO    ] discord.gateway: Shard ID None has successfully RESUMED session 83da4627aecf5665ab7479e3a078cd8d.
